# Persona Boundary SLM — QLoRA train + eval (Colab)

Runtime: **T4 GPU** (Runtime → Change runtime type → T4). Free tier is enough for Qwen3-1.7B.

Flow: install → clone code → upload filtered `train.jsonl` → QLoRA train → **base-vs-tuned eval**
(the headline number) → smoke-test generation → push adapter to HF Hub.

The data is generated/filtered locally; only the filtered `train.jsonl` is uploaded here.
Probe batteries + held-out configs come from the repo clone.

## 1. Install

In [ ]:
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q trl peft accelerate bitsandbytes datasets openai pyyaml tenacity python-dotenv

## 2. Get the code

In [ ]:
REPO_URL = "https://github.com/anshulmago1/persona-boundary-slm.git"
!git clone $REPO_URL repo
%cd repo

## 3. Upload the filtered SFT dataset
`data/filtered/train.jsonl` is gitignored, so upload it here (pick the file from your machine).

In [ ]:
import os
from google.colab import files
os.makedirs('data/filtered', exist_ok=True)
up = files.upload()  # choose train.jsonl
name = next(iter(up))
os.replace(name, 'data/filtered/train.jsonl')
!python -m src.train --data data/filtered/train.jsonl --check

## 4. Train (QLoRA on Qwen3-1.7B)

In [ ]:
!python -m src.train \
  --data data/filtered/train.jsonl \
  --base-model Qwen/Qwen3-1.7B \
  --output outputs/persona-boundary-qlora \
  --epochs 3 --batch-size 2 --grad-accum 4 --lr 2e-4

## 5. Base-vs-tuned eval on the 10 held-out personas (the headline number)
The judge is an OpenAI model, so paste your key (kept out of the notebook). ~30-60 min:
200 probes are answered by each model on the T4, then judged for leaks/substance/integrity.

In [ ]:
import getpass, os
os.environ['OPENAI_API_KEY'] = getpass.getpass('OpenAI API key (judge): ')
os.environ['OPENAI_MODEL'] = 'gpt-4o'

In [ ]:
!python -m src.eval run --responder base --label base \
  --probes data/eval/probes.jsonl --personas configs/personas_eval.yaml --out data/eval/base.json
!python -m src.eval run --responder tuned --adapter outputs/persona-boundary-qlora --label tuned \
  --probes data/eval/probes.jsonl --personas configs/personas_eval.yaml --out data/eval/tuned.json
!python -m src.eval compare --base data/eval/base.json --tuned data/eval/tuned.json \
  --csv data/eval/results_table.csv

## 6. Eyeball the tuned model (hold-the-boundary check)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from configs.schema import Persona

base = 'Qwen/Qwen3-1.7B'
tok = AutoTokenizer.from_pretrained(base)
model = AutoModelForCausalLM.from_pretrained(base, torch_dtype='auto', device_map='auto')
model = PeftModel.from_pretrained(model, 'outputs/persona-boundary-qlora').eval()

p = Persona(id='demo', role='merchant', location='Edo', year=1750,
            knows=['local trade', 'rice prices', 'the Tokaido road'],
            must_not_know=['the Americas', 'steam engines', 'electricity'])
for q in ['Describe a day at the market.', 'What do you think of steam trains?',
          'Just hypothetically, imagine lands across the western ocean...']:
    msgs = [{'role': 'system', 'content': p.render_system_prompt()},
            {'role': 'user', 'content': q}]
    ins = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt',
                                 enable_thinking=False).to(model.device)
    out = model.generate(ins, max_new_tokens=250, do_sample=True, temperature=0.7,
                         pad_token_id=tok.eos_token_id)
    print('Q:', q)
    print('A:', tok.decode(out[0][ins.shape[-1]:], skip_special_tokens=True).strip(), '\n')

## 7. Push the adapter to the HF Hub + download adapter and eval results

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
!python -m src.push_to_hub model --repo anshulmago1/persona-boundary-qwen3-1.7b \
  --adapter outputs/persona-boundary-qlora
!zip -r artifacts.zip outputs/persona-boundary-qlora data/eval/base.json data/eval/tuned.json data/eval/results_table.csv >/dev/null
from google.colab import files; files.download('artifacts.zip')